# Usando RAG local para análise de um artigo científico


- Constrói RAG especialista no artigo em PDF usado no input

- Realiza testes

In [ ]:
import os

import pandas as pd
import numpy as np

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
import torch

import faiss

from IPython.display import Markdown, display

In [ ]:
MODEL_PATH = r'/home/msc/Downloads/hf_models'

EMBEDDING_MODEL_NAME = 'Qwen/Qwen3-Embedding-0.6B'

LLM_MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'

## Embedding

In [3]:
index = faiss.read_index('../data/processed/chunks.index')
df_chunks_with_embeddings = pd.read_parquet('../data/processed/chunks_with_embeddings.parquet')

In [4]:
tokenizer = AutoTokenizer.from_pretrained(f'{MODEL_PATH}/{EMBEDDING_MODEL_NAME}', trust_remote_code=True)
model = AutoModel.from_pretrained(f'{MODEL_PATH}/{EMBEDDING_MODEL_NAME}', trust_remote_code=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device, dtype=torch.float32)
model.eval()

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Qwen3Model(
  (embed_tokens): Embedding(151669, 1024)
  (layers): ModuleList(
    (0-27): 28 x Qwen3DecoderLayer(
      (self_attn): Qwen3Attention(
        (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
        (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
        (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
      )
      (mlp): Qwen3MLP(
        (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
        (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
        (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
      (post_attention_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
    )
  )
  (norm): Qwen3RM

In [ ]:
def embed_texts(texts, batch_size=32, max_length=1024):
    all_embeddings = []

    for i in range(0, len(texts)):
        batch = texts[i: i + batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            emb = outputs.last_hidden_state.mean(dim=1)

        all_embeddings.append(emb.cpu().numpy())

    return np.vstack(all_embeddings)

In [ ]:
def embed_query(text):
    emb = embed_texts([text], batch_size=1)
    faiss.normalize_L2(emb)
    return emb


def retrieve(query, k=5):
    q_emb = embed_query(query)

    x, y = index.search(q_emb, k)
    results = df_chunks_with_embeddings.iloc[x[0]]

    return results['chunk'].tolist()

## RAG

In [7]:
tokenizer_llm = AutoTokenizer.from_pretrained(
    pretrained_model_name_or_path=f'{MODEL_PATH}/{LLM_MODEL_NAME}'
)

### Prompt

In [8]:
def build_prompt(question, contexts):
    context_text = '\n\n'.join(contexts)

    messages = [
        {
            'role': 'system',
            'content': (
                'You are a helpful research assistant who answers user questions based on the provided context.\n'
                'Follow these rules:\n'
                f'1. Answer using *only* the provided Context:{context_text}.\n'
                '2. Check whether the question is related to the provided context.\n'
                "3. If the question is not related to the *context*, reply *I don't know about this*.\n"
                '4. Make your answer concise and do not include explanations.'
            )
        },
        {
            'role': 'user',
            'content': (
                f'Question:\n{question}\n\n'
            )
        }
    ]

    text = tokenizer_llm.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return text

### LLM

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4'
)
model_llm = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=f'{MODEL_PATH}/{LLM_MODEL_NAME}',
    quantization_config=bnb_config,
    device_map='auto'
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [10]:
def generate_answer(prompt, max_tokens=5*512):
    model_inputs = tokenizer_llm(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=max_tokens
    ).to(model_llm.device)

    with torch.no_grad():
        outputs = model_llm.generate(
            **model_inputs,
            max_new_tokens=100,
            do_sample=False,
            eos_token_id=tokenizer_llm.eos_token_id,  # termina ao fim do texto
            pad_token_id=tokenizer_llm.pad_token_id
        )

    generated_tokens = outputs[0][model_inputs['input_ids'].shape[1]:]

    return tokenizer_llm.decode(generated_tokens, skip_special_tokens=True).strip()

In [ ]:
def rag(question, k=1):
    context = retrieve(question, k=k)

    display(Markdown(f'🗂️ **Retrieved context:** {context}'))

    text = build_prompt(question, context)
    
    answer = generate_answer(text)

    return answer

## Sessão de Q&A

In [12]:
# Questions:
# ==========
#
# How do we quantify an agent's homophily?
# Who is the best soccer player of all time?
# What are the main references of the article?
# What does *agent* mean in the context of the article?


In [14]:
print('===== Entre com suas questões (pressione Enter para sair) =====')
while 1 > 0:
    question = input('Humano: ')
    if not len(question):
        break
    else:
        display(Markdown(f'🤔 **Pergunta:**  *_{question}_* <br/><br/>'))
        answer = rag(question=question)
        display(Markdown(f'🤖 **Resposta:**  {answer} <br/><br/>'))
        display(Markdown(('---')))

===== Entre com suas questões (pressione Enter para sair) =====

🤔 **Pergunta:**  *_How do we quantify an agent's homophily?_* <br/><br/>

🗂️ **Retrieved context:** ['. In contrast, when there is a signiﬁcant probability of misinformation, agents will be uncertain about how to interpret articles that disagree with their priors and this may place an upper bound on the speed and possibility of learning (see Acemoglu et al. (2016)). Despite its simplicity, our model makes several new empirical predictions, most notably related to the non-monotonic effects of homophily and polarization and to platform incentives and algorithmic decisions']

🤖 **Resposta:**  To quantify an agent's homophily, one can use measures such as the degree of similarity between individuals or the frequency of interactions between them. These metrics help in understanding how similar or dissimilar different groups of people are, which is a key aspect of homophily. <br/><br/>

---

...

---

🤔 **Pergunta:**  *_What does *agent* mean in the context of the article?_* <br/><br/>

🗂️ **Retrieved context:** ['. Given this provenance policy, we now consider agent 2, who next receives the article. Agent 2 does not know whether the article has already been fact-checked by agent 1, but knows the provenance policy is in place. Given the article was passed to her, she assesses that there is a higher probability that the article is truthful relative to the case with no provenance policy']

🤖 **Resposta:**  Agent means someone who assists or helps another person achieve something. In this context, it refers to the article's author or publisher. <br/><br/>

---